<div style="display: flex; align-items: center; gap: 15px;">
    <img src="https://cdn-icons-png.flaticon.com/128/2737/2737448.png" alt="Backtesting Logo" title="Backtesting Logo" width="64" height="64">
    <h1>Backtesting Notebook</h1>
</div>
<hr style="border: 1px solid #ccc; margin: 10px 0;">

### Import 



In [ ]:
import warnings
warnings.filterwarnings("ignore")

from igtrader.Strategies.Helpers import load_data
from igtrader.Strategies.BuyTrendFollowing import BuyTrendFollowingBA
from igtrader.Strategies.SellTrendFollowing import SellTrendFollowingBA
from igtrader.backtestingpy.backtesting.backtesting import Backtest
import pandas as pd

### Run backtest

<details>
<summary>Click to see parameter descriptions</summary>

| Parameter | Description | Available Options |
|-----------|-------------|-------------------|
| **symbol_to_trade** | The trading symbol/asset | `NDX` • `AAPL` • `EURUSD` |
| **backtest_duration** | Time period for backtest | `d` (day) • `w` (week) • `m` (month) • `y` (year) e.g. `10d`,`2w`,`1m`,`3y`|
| **time_unit** | Interval between data points | `20secs` • `1min` • `1h` • `1d` • `1w` |
| **end_date** | End date for the backtest | `DD/MM/YYYY` | 
| **timezone** | Timezone for data | `UTC` • `Europe/Paris` • `America/New_York` |
| **spread** | Defines broker spread in percentage of the symbol price | |

</details>

In [ ]:
# -----------Define parameters for the backtest----------------
symbol_to_trade = 'NDX' 
backtest_duration = '5d' 
time_unit = '20secs'
end_date = '01/04/2025' 
timezone='Europe/Paris'
spread = 0.0002
save_results = False

# -----------Load data and run the backtest--------------------
data = load_data(symbol=symbol_to_trade, period=backtest_duration, interval=time_unit, end_date=end_date, timezone=timezone)
strategy = BuyTrendFollowingBA
bt = Backtest(data, strategy, cash=100000, commission=.00, spread=spread, exclusive_orders=True)

stats = bt.run()
print(stats)
if save_results:
    df = stats.to_frame()
    for col in df.select_dtypes(include=['timedelta64']).columns:
        df[col] = df[col].dt.total_seconds() 
        
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str)
        
    df.to_parquet(f"../backtest_results/{symbol_to_trade}_{backtest_duration}_{time_unit}_BuyTrendFollowingBA.parquet")

In [ ]:
display(data.head())
pd.set_option('display.max_columns', None)  # Affiche toutes les colonnes
display(stats['_trades'])

### Plot backtest results 

<details>
<summary>Click to see parameter descriptions</summary>

| Parameter | Description | Available Options |
|-----------|-------------|-------------------|
| **resample** | Reduces candle time units on plot for better performance | `False` • `True` |
| **ohlc_height** | Height of the main windows with candles | `Integer` |
| **indicator_height** | Height of the indicator window |  `Integer` |
| **plot_volume** | Defines if the volumes are plotted or not | `False` • `True` |
  

</details>

In [ ]:
bt.plot(resample=False, 
        ohlc_height=700, 
        indicator_height=300,
        plot_volume=False, 
        plot_equity=True,
        superimpose=False)